# CodeGen0.1

Notebook ini adalah versi `.ipynb` dari `CodeGen0.1.py` untuk training XGBoost forecasting solar irradiance 24 jam ke depan per region representatif di Indonesia.

Jalankan cell code di bawah untuk menjalankan pipeline penuh semua region.


In [ ]:
from __future__ import annotations

import json
import traceback
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


SEED = 42
DATA_PATH = Path("nasa_dataset_daily_20200101_20260101_representative_point_area.csv")
ARTIFACTS_ROOT = Path("artifacts_xgboost_representative_points")

TARGET_COL = "ALLSKY_SFC_SW_DWN"
MISSING_SENTINEL = -999.0
LOOKBACK = 24
HORIZON = 24
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAPE_EPSILON = 1.0

BASE_FEATURES = [
    "CLOUD_AMT",
    "T2M",
    "RH2M",
    "PS",
    "CLRSKY_SFC_SW_DWN",
]

CANONICAL_REGIONS = [
    "Sumatra",
    "Jawa",
    "Kalimantan",
    "Sulawesi",
    "Nusa Tenggara",
    "Maluku",
    "Papua",
]

REGION_NAME_MAP = {
    "sumatra": "Sumatra",
    "sumatera": "Sumatra",
    "jawa": "Jawa",
    "kalimantan": "Kalimantan",
    "sulawesi": "Sulawesi",
    "nusa tenggara": "Nusa Tenggara",
    "bali_nusa": "Nusa Tenggara",
    "ntb_ntt": "Nusa Tenggara",
    "maluku": "Maluku",
    "papua": "Papua",
}

REPRESENTATIVE_COORDS = {
    "Sumatra": {"latitude": None, "longitude": None},
    "Jawa": {"latitude": None, "longitude": None},
    "Kalimantan": {"latitude": None, "longitude": None},
    "Sulawesi": {"latitude": None, "longitude": None},
    "Nusa Tenggara": {"latitude": None, "longitude": None},
    "Maluku": {"latitude": None, "longitude": None},
    "Papua": {"latitude": None, "longitude": None},
}

XGB_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "random_state": SEED,
    "n_jobs": -1,
}


def normalize_region_name(name: str) -> str | None:
    if pd.isna(name):
        return None
    cleaned = str(name).strip().lower()
    return REGION_NAME_MAP.get(cleaned)


def region_to_folder_name(region_name: str) -> str:
    return region_name.replace(" ", "_")


def load_dataset(csv_path: Path) -> pd.DataFrame:
    print(f"[INFO] Loading dataset: {csv_path}")
    df = pd.read_csv(csv_path)
    return df


def build_datetime(df: pd.DataFrame) -> pd.DataFrame:
    required_cols = ["YEAR", "MO", "DY", "HR"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing datetime columns: {missing}")

    out = df.copy()
    out["datetime"] = pd.to_datetime(
        {
            "year": out["YEAR"],
            "month": out["MO"],
            "day": out["DY"],
            "hour": out["HR"],
        },
        errors="coerce",
    )
    out = out.dropna(subset=["datetime"]).copy()
    return out


def clean_region_data(df: pd.DataFrame, region_name: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    region_df = df.loc[df["REP_NAME"] == region_name].copy()
    if region_df.empty:
        raise ValueError(f"No rows found for region: {region_name}")

    region_df = region_df.sort_values("datetime").reset_index(drop=True)

    numeric_cols = BASE_FEATURES + [TARGET_COL]
    for col in numeric_cols:
        region_df[col] = pd.to_numeric(region_df[col], errors="coerce")
        region_df.loc[region_df[col] == MISSING_SENTINEL, col] = np.nan

    raw_rows = len(region_df)
    dropped_target_missing = int(region_df[TARGET_COL].isna().sum())
    region_df = region_df.dropna(subset=[TARGET_COL]).copy()

    missing_before_ffill = region_df[BASE_FEATURES].isna().sum().to_dict()
    region_df[BASE_FEATURES] = region_df[BASE_FEATURES].ffill()
    missing_after_ffill = region_df[BASE_FEATURES].isna().sum().to_dict()

    info = {
        "raw_rows": raw_rows,
        "rows_after_target_drop": int(len(region_df)),
        "dropped_target_missing": dropped_target_missing,
        "missing_before_ffill": missing_before_ffill,
        "missing_after_ffill": missing_after_ffill,
    }
    return region_df, info


def create_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["sin_hour"] = np.sin(2 * np.pi * out["HR"] / 24.0)
    out["cos_hour"] = np.cos(2 * np.pi * out["HR"] / 24.0)
    out["sin_month"] = np.sin(2 * np.pi * out["MO"] / 12.0)
    out["cos_month"] = np.cos(2 * np.pi * out["MO"] / 12.0)
    return out


def create_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    target_lags = list(range(1, LOOKBACK + 1))
    feature_lags = [1, 2, 3, 6, 12, 24]

    for lag in target_lags:
        out[f"{TARGET_COL}_lag_{lag}"] = out[TARGET_COL].shift(lag)

    for feature in BASE_FEATURES:
        for lag in feature_lags:
            out[f"{feature}_lag_{lag}"] = out[feature].shift(lag)

    rolling_sources = [TARGET_COL, "CLOUD_AMT", "CLRSKY_SFC_SW_DWN"]
    for feature in rolling_sources:
        shifted = out[feature].shift(1)
        out[f"{feature}_roll_mean_3"] = shifted.rolling(window=3, min_periods=3).mean()
        out[f"{feature}_roll_mean_6"] = shifted.rolling(window=6, min_periods=6).mean()
        out[f"{feature}_roll_min_6"] = shifted.rolling(window=6, min_periods=6).min()
        out[f"{feature}_roll_max_6"] = shifted.rolling(window=6, min_periods=6).max()

    return out


def create_multistep_targets(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    target_cols = []
    for step in range(1, HORIZON + 1):
        col = f"target_t_plus_{step}"
        out[col] = out[TARGET_COL].shift(-step)
        target_cols.append(col)

    out["forecast_start"] = out["datetime"].shift(-1)
    return out


def time_split(df: pd.DataFrame, train_ratio: float = TRAIN_RATIO, val_ratio: float = VAL_RATIO) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n_rows = len(df)
    train_end = int(n_rows * train_ratio)
    val_end = int(n_rows * (train_ratio + val_ratio))

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    return train_df, val_df, test_df


def apply_train_median_imputation(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, float]]:
    train_out = train_df.copy()
    val_out = val_df.copy()
    test_out = test_df.copy()

    medians = train_out[feature_cols].median(numeric_only=True).to_dict()

    train_out[feature_cols] = train_out[feature_cols].fillna(medians)
    val_out[feature_cols] = val_out[feature_cols].fillna(medians)
    test_out[feature_cols] = test_out[feature_cols].fillna(medians)

    return train_out, val_out, test_out, medians


def prepare_supervised_region_data(region_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    processed = create_time_features(region_df)
    processed = create_lag_features(processed)
    processed = create_multistep_targets(processed)

    target_cols = [f"target_t_plus_{step}" for step in range(1, HORIZON + 1)]
    base_feature_cols = BASE_FEATURES + ["sin_hour", "cos_hour", "sin_month", "cos_month"]

    lag_feature_cols = [
        col
        for col in processed.columns
        if col.endswith(tuple([f"_lag_{lag}" for lag in range(1, LOOKBACK + 1)]))
        or "_roll_" in col
    ]
    feature_cols = base_feature_cols + lag_feature_cols

    required_history_cols = [f"{TARGET_COL}_lag_{LOOKBACK}"] + [f"{feature}_lag_24" for feature in BASE_FEATURES]
    processed = processed.dropna(subset=required_history_cols + ["forecast_start"]).copy()
    processed = processed.dropna(subset=target_cols).copy()
    processed = processed.sort_values("datetime").reset_index(drop=True)

    return processed, feature_cols, target_cols


def post_process_predictions(
    pred: np.ndarray,
    forecast_starts: pd.Series,
    apply_night_zeroing: bool = False,
    night_hours: tuple[int, int] = (18, 5),
) -> np.ndarray:
    pred = np.clip(pred, 0, None)

    if not apply_night_zeroing:
        return pred

    processed = pred.copy()
    start_hour, end_hour = night_hours

    for row_idx, forecast_start in enumerate(forecast_starts):
        horizon_times = pd.date_range(start=forecast_start, periods=HORIZON, freq="h")
        for step_idx, ts in enumerate(horizon_times):
            hour = ts.hour
            is_night = hour >= start_hour or hour <= end_hour
            if is_night:
                processed[row_idx, step_idx] = 0.0

    return processed


def train_xgboost_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_cols: list[str],
    apply_night_zeroing: bool = False,
) -> dict[str, Any]:
    x_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    x_val = val_df[feature_cols].to_numpy(dtype=np.float32)
    x_test = test_df[feature_cols].to_numpy(dtype=np.float32)

    y_train = train_df[target_cols].to_numpy(dtype=np.float32)
    y_val = val_df[target_cols].to_numpy(dtype=np.float32)
    y_test = test_df[target_cols].to_numpy(dtype=np.float32)

    models: list[XGBRegressor] = []
    train_rmse_histories: list[list[float]] = []
    val_rmse_histories: list[list[float]] = []

    print(f"[INFO] Training {HORIZON} horizon-specific XGBoost models")

    for horizon_idx in range(HORIZON):
        target_name = target_cols[horizon_idx]
        print(f"  - Horizon {horizon_idx + 1:02d}/{HORIZON}: {target_name}")

        model = XGBRegressor(**XGB_PARAMS)
        model.fit(
            x_train,
            y_train[:, horizon_idx],
            eval_set=[
                (x_train, y_train[:, horizon_idx]),
                (x_val, y_val[:, horizon_idx]),
            ],
            verbose=False,
        )

        evals_result = model.evals_result()
        train_rmse_histories.append(evals_result["validation_0"]["rmse"])
        val_rmse_histories.append(evals_result["validation_1"]["rmse"])
        models.append(model)

    y_pred = np.column_stack([model.predict(x_test) for model in models]).astype(np.float32)
    y_pred = post_process_predictions(
        y_pred,
        forecast_starts=test_df["forecast_start"],
        apply_night_zeroing=apply_night_zeroing,
    )

    return {
        "models": models,
        "y_true": y_test,
        "y_pred": y_pred,
        "train_rmse_history": np.mean(np.array(train_rmse_histories, dtype=np.float32), axis=0),
        "val_rmse_history": np.mean(np.array(val_rmse_histories, dtype=np.float32), axis=0),
    }


def evaluate_forecast(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, Any]:
    rmse_all = float(np.sqrt(mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1))))
    mae_all = float(mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1)))
    r2_all = float(r2_score(y_true.reshape(-1), y_pred.reshape(-1)))

    rmse_per_horizon = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    mae_per_horizon = np.mean(np.abs(y_true - y_pred), axis=0)
    r2_per_horizon = [
        float(r2_score(y_true[:, step_idx], y_pred[:, step_idx]))
        for step_idx in range(y_true.shape[1])
    ]

    mask_all = np.abs(y_true.reshape(-1)) > MAPE_EPSILON
    if mask_all.any():
        mape_all = float(
            np.mean(
                np.abs(
                    (y_true.reshape(-1)[mask_all] - y_pred.reshape(-1)[mask_all])
                    / y_true.reshape(-1)[mask_all]
                )
            )
            * 100.0
        )
    else:
        mape_all = float("nan")

    mape_per_horizon = []
    for step_idx in range(y_true.shape[1]):
        y_true_step = y_true[:, step_idx]
        y_pred_step = y_pred[:, step_idx]
        mask_step = np.abs(y_true_step) > MAPE_EPSILON
        if mask_step.any():
            mape_step = float(
                np.mean(np.abs((y_true_step[mask_step] - y_pred_step[mask_step]) / y_true_step[mask_step]))
                * 100.0
            )
        else:
            mape_step = float("nan")
        mape_per_horizon.append(mape_step)

    return {
        "rmse_all_horizon": rmse_all,
        "mae_all_horizon": mae_all,
        "r2_all_horizon": r2_all,
        "mape_all_horizon_pct": mape_all,
        "rmse_per_horizon": rmse_per_horizon.tolist(),
        "mae_per_horizon": mae_per_horizon.tolist(),
        "r2_per_horizon": r2_per_horizon,
        "mape_per_horizon_pct": mape_per_horizon,
        "mape_epsilon": MAPE_EPSILON,
    }


def plot_forecast_sample(
    region_name: str,
    forecast_start: pd.Timestamp,
    y_true_sample: np.ndarray,
    y_pred_sample: np.ndarray,
    save_path: Path,
) -> None:
    horizon_hours = np.arange(1, HORIZON + 1)
    plt.figure(figsize=(12, 5))
    plt.plot(horizon_hours, y_true_sample, marker="o", label="Actual")
    plt.plot(horizon_hours, y_pred_sample, marker="o", label="Predicted")
    plt.title(f"Forecast Sample 24 Jam - {region_name}")
    plt.xlabel("Jam ke depan")
    plt.ylabel(TARGET_COL)
    plt.xticks(range(1, HORIZON + 1, 2))
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_rmse_per_horizon(region_name: str, rmse_per_horizon: list[float], save_path: Path) -> None:
    horizon_hours = np.arange(1, HORIZON + 1)
    plt.figure(figsize=(12, 5))
    plt.plot(horizon_hours, rmse_per_horizon, marker="o")
    plt.title(f"RMSE per Horizon - {region_name}")
    plt.xlabel("Horizon (jam)")
    plt.ylabel("RMSE")
    plt.xticks(range(1, HORIZON + 1, 2))
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_training_history(region_name: str, train_history: np.ndarray, val_history: np.ndarray, save_path: Path) -> None:
    boosting_rounds = np.arange(1, len(train_history) + 1)
    plt.figure(figsize=(10, 4))
    plt.plot(boosting_rounds, train_history, label="train_rmse")
    plt.plot(boosting_rounds, val_history, label="val_rmse")
    plt.title(f"Training History - {region_name}")
    plt.xlabel("Boosting round")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def save_feature_importance(region_dir: Path, feature_cols: list[str], models: list[XGBRegressor]) -> Path:
    importances = np.array([model.feature_importances_ for model in models], dtype=np.float32)
    avg_importance = importances.mean(axis=0)

    importance_df = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance_mean": avg_importance,
        }
    ).sort_values("importance_mean", ascending=False)

    csv_path = region_dir / "feature_importance.csv"
    png_path = region_dir / "feature_importance_top20.png"
    importance_df.to_csv(csv_path, index=False)

    top_k = importance_df.head(20).iloc[::-1]
    plt.figure(figsize=(10, 8))
    plt.barh(top_k["feature"], top_k["importance_mean"])
    plt.title("Top 20 Feature Importance")
    plt.xlabel("Mean importance")
    plt.tight_layout()
    plt.savefig(png_path, dpi=150)
    plt.close()

    return csv_path


def save_artifacts(
    region_name: str,
    region_info: dict[str, Any],
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_cols: list[str],
    medians: dict[str, float],
    train_result: dict[str, Any],
    metrics: dict[str, Any],
) -> dict[str, str]:
    region_dir = ARTIFACTS_ROOT / region_to_folder_name(region_name)
    region_dir.mkdir(parents=True, exist_ok=True)

    models_path = region_dir / "xgboost_models.joblib"
    preprocessor_path = region_dir / "preprocessor.joblib"
    metrics_path = region_dir / "metrics.json"
    prediction_sample_path = region_dir / "prediction_sample.csv"
    forecast_plot_path = region_dir / "forecast_sample.png"
    rmse_plot_path = region_dir / "rmse_per_horizon.png"
    history_plot_path = region_dir / "training_history.png"
    summary_json_path = region_dir / "summary.json"
    summary_txt_path = region_dir / "summary.txt"

    joblib.dump(train_result["models"], models_path)
    joblib.dump(
        {
            "feature_columns": feature_cols,
            "target_columns": target_cols,
            "train_feature_medians": medians,
            "lookback": LOOKBACK,
            "horizon": HORIZON,
            "base_features": BASE_FEATURES,
            "post_processing": {
                "clip_non_negative": True,
                "night_zeroing_available": True,
                "night_zeroing_default": False,
            },
        },
        preprocessor_path,
    )

    with metrics_path.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    sample_idx = -1
    y_true_sample = train_result["y_true"][sample_idx]
    y_pred_sample = train_result["y_pred"][sample_idx]
    forecast_start = pd.to_datetime(test_df.iloc[sample_idx]["forecast_start"])
    forecast_timestamps = pd.date_range(start=forecast_start, periods=HORIZON, freq="h")

    prediction_sample_df = pd.DataFrame(
        {
            "timestamp": forecast_timestamps,
            "horizon_hour": np.arange(1, HORIZON + 1),
            "actual": y_true_sample,
            "predicted": y_pred_sample,
        }
    )
    prediction_sample_df.to_csv(prediction_sample_path, index=False)

    plot_forecast_sample(region_name, forecast_start, y_true_sample, y_pred_sample, forecast_plot_path)
    plot_rmse_per_horizon(region_name, metrics["rmse_per_horizon"], rmse_plot_path)
    plot_training_history(region_name, train_result["train_rmse_history"], train_result["val_rmse_history"], history_plot_path)
    save_feature_importance(region_dir, feature_cols, train_result["models"])

    summary = {
        "region_name": region_name,
        "artifact_dir": str(region_dir.resolve()),
        "raw_rows": region_info["raw_rows"],
        "rows_after_target_drop": region_info["rows_after_target_drop"],
        "dropped_target_missing": region_info["dropped_target_missing"],
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_range": [str(train_df["datetime"].min()), str(train_df["datetime"].max())],
        "val_range": [str(val_df["datetime"].min()), str(val_df["datetime"].max())],
        "test_range": [str(test_df["datetime"].min()), str(test_df["datetime"].max())],
        "feature_count": len(feature_cols),
        "target_count": len(target_cols),
        "metrics": metrics,
        "coords_template": REPRESENTATIVE_COORDS.get(region_name),
        "xgb_params": XGB_PARAMS,
    }

    with summary_json_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    summary_lines = [
        f"Region           : {region_name}",
        f"Artifact dir     : {region_dir.resolve()}",
        f"Raw rows         : {region_info['raw_rows']}",
        f"Rows after y drop: {region_info['rows_after_target_drop']}",
        f"Dropped y missing: {region_info['dropped_target_missing']}",
        f"Train rows       : {len(train_df)}",
        f"Val rows         : {len(val_df)}",
        f"Test rows        : {len(test_df)}",
        f"RMSE all horizon : {metrics['rmse_all_horizon']:.6f}",
        f"MAE all horizon  : {metrics['mae_all_horizon']:.6f}",
        f"R2 all horizon   : {metrics['r2_all_horizon']:.6f}",
        f"MAPE all horizon : {metrics['mape_all_horizon_pct']:.6f}%",
    ]
    summary_txt_path.write_text("\n".join(summary_lines), encoding="utf-8")

    return {
        "region_dir": str(region_dir.resolve()),
        "models_path": str(models_path.resolve()),
        "metrics_path": str(metrics_path.resolve()),
        "prediction_sample_path": str(prediction_sample_path.resolve()),
        "summary_json_path": str(summary_json_path.resolve()),
    }


def process_single_region(df: pd.DataFrame, region_name: str, apply_night_zeroing: bool = False) -> dict[str, Any]:
    print(f"\n[INFO] ===== Processing region: {region_name} =====")

    region_df, region_info = clean_region_data(df, region_name)
    supervised_df, feature_cols, target_cols = prepare_supervised_region_data(region_df)

    train_df, val_df, test_df = time_split(supervised_df)
    train_df, val_df, test_df, medians = apply_train_median_imputation(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        feature_cols=feature_cols,
    )

    train_result = train_xgboost_model(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        feature_cols=feature_cols,
        target_cols=target_cols,
        apply_night_zeroing=apply_night_zeroing,
    )

    metrics = evaluate_forecast(train_result["y_true"], train_result["y_pred"])
    artifact_paths = save_artifacts(
        region_name=region_name,
        region_info=region_info,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        feature_cols=feature_cols,
        target_cols=target_cols,
        medians=medians,
        train_result=train_result,
        metrics=metrics,
    )

    return {
        "region_name": region_name,
        "status": "success",
        "rmse_all_horizon": metrics["rmse_all_horizon"],
        "mae_all_horizon": metrics["mae_all_horizon"],
        "r2_all_horizon": metrics["r2_all_horizon"],
        "mape_all_horizon_pct": metrics["mape_all_horizon_pct"],
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df),
        "artifact_dir": artifact_paths["region_dir"],
    }


def print_summary_table(summary_df: pd.DataFrame) -> None:
    if summary_df.empty:
        print("[WARN] No successful region runs to summarize.")
        return

    print("\n[INFO] ===== Summary Table =====")
    display_cols = [
        "region_name",
        "status",
        "rmse_all_horizon",
        "mae_all_horizon",
        "r2_all_horizon",
        "mape_all_horizon_pct",
        "train_rows",
        "val_rows",
        "test_rows",
    ]
    print(summary_df[display_cols].to_string(index=False))

    successful = summary_df.loc[summary_df["status"] == "success"].copy()
    if successful.empty:
        print("[WARN] No successful models found.")
        return

    best_rmse_row = successful.loc[successful["rmse_all_horizon"].idxmin()]
    worst_rmse_row = successful.loc[successful["rmse_all_horizon"].idxmax()]
    best_mae_row = successful.loc[successful["mae_all_horizon"].idxmin()]
    worst_mae_row = successful.loc[successful["mae_all_horizon"].idxmax()]
    best_r2_row = successful.loc[successful["r2_all_horizon"].idxmax()]
    worst_r2_row = successful.loc[successful["r2_all_horizon"].idxmin()]
    best_mape_row = successful.loc[successful["mape_all_horizon_pct"].idxmin()]
    worst_mape_row = successful.loc[successful["mape_all_horizon_pct"].idxmax()]

    print("\n[INFO] ===== Best / Worst Region =====")
    print(
        f"Best RMSE  : {best_rmse_row['region_name']} "
        f"({best_rmse_row['rmse_all_horizon']:.6f})"
    )
    print(
        f"Worst RMSE : {worst_rmse_row['region_name']} "
        f"({worst_rmse_row['rmse_all_horizon']:.6f})"
    )
    print(
        f"Best MAE   : {best_mae_row['region_name']} "
        f"({best_mae_row['mae_all_horizon']:.6f})"
    )
    print(
        f"Worst MAE  : {worst_mae_row['region_name']} "
        f"({worst_mae_row['mae_all_horizon']:.6f})"
    )
    print(
        f"Best R2    : {best_r2_row['region_name']} "
        f"({best_r2_row['r2_all_horizon']:.6f})"
    )
    print(
        f"Worst R2   : {worst_r2_row['region_name']} "
        f"({worst_r2_row['r2_all_horizon']:.6f})"
    )
    print(
        f"Best MAPE  : {best_mape_row['region_name']} "
        f"({best_mape_row['mape_all_horizon_pct']:.6f}%)"
    )
    print(
        f"Worst MAPE : {worst_mape_row['region_name']} "
        f"({worst_mape_row['mape_all_horizon_pct']:.6f}%)"
    )


def run_all_regions(
    csv_path: Path = DATA_PATH,
    artifacts_root: Path = ARTIFACTS_ROOT,
    apply_night_zeroing: bool = False,
) -> pd.DataFrame:
    np.random.seed(SEED)
    artifacts_root.mkdir(parents=True, exist_ok=True)
    (artifacts_root / "rep_name_coordinates_template.json").write_text(
        json.dumps(REPRESENTATIVE_COORDS, indent=2),
        encoding="utf-8",
    )

    df = load_dataset(csv_path)
    df = build_datetime(df)

    if "REP_NAME" not in df.columns:
        raise ValueError("Column REP_NAME is required.")

    df["REP_NAME"] = df["REP_NAME"].map(normalize_region_name)
    df = df.dropna(subset=["REP_NAME"]).copy()
    df = df.sort_values(["REP_NAME", "datetime"]).reset_index(drop=True)

    print("[INFO] Available normalized regions:")
    print(df["REP_NAME"].value_counts().sort_index().to_string())

    results: list[dict[str, Any]] = []

    for region_name in CANONICAL_REGIONS:
        try:
            if region_name not in set(df["REP_NAME"].unique()):
                print(f"[WARN] Region not found in dataset after normalization: {region_name}")
                results.append(
                    {
                        "region_name": region_name,
                        "status": "missing",
                        "rmse_all_horizon": np.nan,
                        "mae_all_horizon": np.nan,
                        "r2_all_horizon": np.nan,
                        "mape_all_horizon_pct": np.nan,
                        "train_rows": 0,
                        "val_rows": 0,
                        "test_rows": 0,
                        "artifact_dir": "",
                    }
                )
                continue

            region_result = process_single_region(
                df=df,
                region_name=region_name,
                apply_night_zeroing=apply_night_zeroing,
            )
            results.append(region_result)

        except Exception as exc:
            print(f"[ERROR] Region failed: {region_name}")
            print(f"[ERROR] {exc}")
            traceback.print_exc()
            results.append(
                {
                    "region_name": region_name,
                    "status": "failed",
                    "rmse_all_horizon": np.nan,
                    "mae_all_horizon": np.nan,
                    "r2_all_horizon": np.nan,
                    "mape_all_horizon_pct": np.nan,
                    "train_rows": 0,
                    "val_rows": 0,
                    "test_rows": 0,
                    "artifact_dir": "",
                    "error_message": str(exc),
                }
            )

    summary_df = pd.DataFrame(results)
    summary_csv_path = artifacts_root / "summary_all_regions.csv"
    summary_json_path = artifacts_root / "summary_all_regions.json"
    summary_df.to_csv(summary_csv_path, index=False)
    summary_df.to_json(summary_json_path, orient="records", indent=2)

    print_summary_table(summary_df)
    print(f"\n[INFO] Artifacts root: {artifacts_root.resolve()}")
    print(f"[INFO] Summary CSV   : {summary_csv_path.resolve()}")
    print(f"[INFO] Summary JSON  : {summary_json_path.resolve()}")

    return summary_df


def main() -> None:
    summary_df = run_all_regions(
        csv_path=DATA_PATH,
        artifacts_root=ARTIFACTS_ROOT,
        apply_night_zeroing=False,
    )

    successful = summary_df.loc[summary_df["status"] == "success"]
    print(f"\n[INFO] Finished. Successful regions: {len(successful)}/{len(CANONICAL_REGIONS)}")


if __name__ == "__main__":
    main()
